In [1]:
from datasets import load_dataset 
dataset= load_dataset("theatticusproject/cuad-qa",revision="refs/convert/parquet")
print (dataset)

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 22450
    })
    test: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 4182
    })
})


In [2]:
sample = dataset["train"][0]
print("title", sample["title"], "\ncontext", sample["context"],"\nquestion", sample["question"],"\nanswer", sample["answers"])

title LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT 
context EXHIBIT 10.6

                              DISTRIBUTOR AGREEMENT

         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between Electric City Corp.,  a Delaware  corporation  ("Company")  and Electric City of Illinois LLC ("Distributor") this 7th day of September, 1999.

                                    RECITALS

         A. The  Company's  Business.  The Company is  presently  engaged in the business  of selling an energy  efficiency  device,  which is  referred to as an "Energy  Saver"  which may be improved  or  otherwise  changed  from its present composition (the "Products").  The Company may engage in the business of selling other  products  or  other  devices  other  than  the  Products,  which  will be considered  Products if Distributor  exercises its options pursuant to Section 7 hereof.

         B. Representations.  As an inducement to the Company to enter into this Agreement,  the  Di

In [3]:
unique_contracts = {}
for row in dataset["train"]:
    Title = row["title"]
    Context = row["context"]
    unique_contracts[Title] = Context

print(len(unique_contracts))


408


In [4]:
print (len(list(unique_contracts.values())[0]))

54290


In [5]:
print(type(list(unique_contracts.values())[0]))
print(list(unique_contracts.keys())[0])
print(list(unique_contracts.values())[0][:200])

<class 'str'>
LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT
EXHIBIT 10.6

                              DISTRIBUTOR AGREEMENT

         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between Electric City Corp.,  a Delaware  corporation  ("Com


In [6]:
print(len(list(unique_contracts.values())[0]))

print(len(list(unique_contracts.values())))

54290
408


In [7]:
lengths = [len(cont) for cont in unique_contracts.values()]

print("average:", sum(lengths)/len(lengths))
print("max:", max(lengths))
print("min:", min(lengths))

average: 53991.71078431373
max: 338211
min: 1081


In [8]:
def chunk_text(text, chunk_size=500, overlap=100):
    words = text.split()
    chunks = []
    start = 0
    while start < len (words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap 
    return chunks 

In [9]:
first_contract = list(unique_contracts.values())[0]
chunks = chunk_text(first_contract, chunk_size=400, overlap=80)

print("chunks number:", len(chunks))
print("First chunk:", chunks[0][:200])
print("The last chunk:", chunks[-1][:200])

chunks number: 19
First chunk: EXHIBIT 10.6 DISTRIBUTOR AGREEMENT THIS DISTRIBUTOR AGREEMENT (the "Agreement") is made by and between Electric City Corp., a Delaware corporation ("Company") and Electric City of Illinois LLC ("Distr
The last chunk: construed according to the laws of the State of Illinois. 7. NEW PRODUCTS 7.1 Right of Option. Should Company introduce other products or devices as contemplated by recital paragraph "A", Distributor 


In [10]:
print (chunks)

['EXHIBIT 10.6 DISTRIBUTOR AGREEMENT THIS DISTRIBUTOR AGREEMENT (the "Agreement") is made by and between Electric City Corp., a Delaware corporation ("Company") and Electric City of Illinois LLC ("Distributor") this 7th day of September, 1999. RECITALS A. The Company\'s Business. The Company is presently engaged in the business of selling an energy efficiency device, which is referred to as an "Energy Saver" which may be improved or otherwise changed from its present composition (the "Products"). The Company may engage in the business of selling other products or other devices other than the Products, which will be considered Products if Distributor exercises its options pursuant to Section 7 hereof. B. Representations. As an inducement to the Company to enter into this Agreement, the Distributor has represented that it has or will have the facilities, personnel, and financial capability to promote the sale and use of Products. As an inducement to Distributor to enter into this Agreeme

In [11]:
all_chunks = []

for title, context in unique_contracts.items():
    contract_chunks = chunk_text(context, chunk_size=400, overlap=80)
    
    for i, chunk in enumerate(contract_chunks):
        chunk_dict = {
            "text": chunk,
            "source": title,
            "chunk_id": f"{title}_{i}"
        }
        all_chunks.append(chunk_dict)

print("The total number of chunks:", len(all_chunks))

The total number of chunks: 10457


In [12]:
import sys 
sys.path.append("..")

from src.ingestion import chunk_text, build_all_chunks, build_chroma_index

In [13]:
all_chunks = build_all_chunks(unique_contracts)
print("Total chunks:", len(all_chunks))

Total chunks: 10457


In [14]:
"""
collection = build_chroma_index(all_chunks)
print ("Index built successfully")
print ("number of items in collection:", collection.count())
"""

'\ncollection = build_chroma_index(all_chunks)\nprint ("Index built successfully")\nprint ("number of items in collection:", collection.count())\n'

In [15]:
import shutil
shutil.rmtree("../chroma_db", ignore_errors=True)

In [16]:
from src.ingestion import build_chroma_index

collection = build_chroma_index(all_chunks)
print("Index built successfully!")
print("Number of items in collection:", collection.count())

2026-08-17 14:09:43.826087991 [W:onnxruntime:Default, device_discovery.cc:134 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


Added batch 0 to 5000
Added batch 5000 to 10000
Added batch 10000 to 15000
Index built successfully!
Number of items in collection: 10457


In [17]:
"""BufferError
collection = build_chroma_index(all_chunks)
print ("Index built successfully")
print ("number of items in collection:", collection.count())
"""

'BufferError\ncollection = build_chroma_index(all_chunks)\nprint ("Index built successfully")\nprint ("number of items in collection:", collection.count())\n'

In [18]:
"""import shutil
shutil.rmtree("../chroma_db", ignore_errors=True)

import importlib
import src.ingestion
importlib.reload(src.ingestion)
from src.ingestion import build_chroma_index

collection = build_chroma_index(all_chunks)
print("Index built successfully!")
print("Number of items in collection:", collection.count())"""

'import shutil\nshutil.rmtree("../chroma_db", ignore_errors=True)\n\nimport importlib\nimport src.ingestion\nimportlib.reload(src.ingestion)\nfrom src.ingestion import build_chroma_index\n\ncollection = build_chroma_index(all_chunks)\nprint("Index built successfully!")\nprint("Number of items in collection:", collection.count())'

In [19]:
import os
print("Current working directory:", os.getcwd())
print("Absolute path to chroma_db:", os.path.abspath("../chroma_db"))

Current working directory: /workspaces/contract-qa-rag/notebooks
Absolute path to chroma_db: /workspaces/contract-qa-rag/chroma_db


In [20]:
import importlib
import src.ingestion
importlib.reload(src.ingestion)
from src.ingestion import build_bm25_index

bm25 = build_bm25_index(all_chunks)
print("BM25 index built successfully!")
print(type(bm25))

BM25 index built successfully!
<class 'rank_bm25.BM25Okapi'>


In [21]:
sample_query = "termination clause"
tokenized_query = sample_query.lower().split()

scores = bm25.get_scores(tokenized_query)
print("Number of scores:", len(scores))
print("Max score:", max(scores))

Number of scores: 10457
Max score: 8.597926219874083


In [22]:
from dotenv import load_dotenv
import os

load_dotenv("../.env")

api_key = os.getenv("GROQ_API_KEY")
print("Key loaded:", api_key is not None)
print("First 5 chars:", api_key[:5] if api_key else "N/A")


Key loaded: True
First 5 chars: gsk_1


In [24]:
import importlib
import src.query_rewriting
importlib.reload(src.query_rewriting)
from src.query_rewriting import rewrite_query

test_query = "شو بند الإنهاء بهاد العقد؟"
rewritten = rewrite_query(test_query, api_key)

print("Original:", test_query)
print("Rewritten:", rewritten)

Original: شو بند الإنهاء بهاد العقد؟
Rewritten: What are the termination provisions stipulated in this agreement?


In [25]:
import importlib
import src.retrieval
importlib.reload(src.retrieval)
from src.retrieval import semantic_search

results = semantic_search(rewritten, collection, n_results=5)
print(results)

{'ids': [['GarrettMotionInc_20181001_8-K_EX-2.4_11364532_EX-2.4_Intellectual Property Agreement_18', 'IMMUNOMEDICSINC_08_07_2019-EX-10.1-PROMOTION AGREEMENT_37', 'ExactSciencesCorp_20180822_8-K_EX-10.1_11331629_EX-10.1_Promotion Agreement_71', 'ReedsInc_20191113_10-Q_EX-10.4_11888303_EX-10.4_Development Agreement_7', 'ACCURAYINC_09_01_2010-EX-10.31-DISTRIBUTOR AGREEMENT_32']], 'embeddings': None, 'documents': [['an agreement in writing signed by a duly authorized officer of each of the Parties. Section 9.02. Termination prior to the Distribution. This Agreement may be terminated by Honeywell at any time, in its sole discretion, prior to the Distribution; provided, however, that this Agreement shall automatically terminate upon the termination of the Separation Agreement in accordance with its terms. Section 9.03. Effect of Termination; Survival. In the event of any termination of this Agreement prior to the Distribution, neither Party (nor any member of their Group or any of their resp

In [26]:
import importlib
import src.retrieval
importlib.reload(src.retrieval)
from src.retrieval import semantic_search, bm25_search

bm25_results = bm25_search(rewritten, bm25, all_chunks, n_results=5)
for chunk in bm25_results:
    print(chunk["source"], "-", chunk["text"][:100])
    print("---")

REWALKROBOTICSLTD_07_10_2014-EX-10.2-STRATEGIC ALLIANCE AGREEMENT - with the contact person for the other party regarding issues arising under this agreement. 7. RELATI
---
InmodeLtd_20190729_F-1A_EX-10.9_11743243_EX-10.9_Manufacturing Agreement - order details. The invoice will be quoted in US Dollars. 14.2 Contractor and Customer agree to terms
---
SENMIAOTECHNOLOGYLTD_02_19_2019-EX-10.5-Collaboration Agreement - is unable to pay while the carriage Agreement losses, or if the passenger requests the Driver User o
---
LECLANCHÉ S.A. - JOINT DEVELOPMENT AND MARKETING AGREEMENT - breach of this undertaking; available to the recipient Party on a non confidential basis from a sour
---
InmodeLtd_20190729_F-1A_EX-10.9_11743243_EX-10.9_Manufacturing Agreement - to compensate Contractor for Products and materials as stipulated in Sections 6 and 8 of this Agreem
---


In [27]:
short_query = "termination clause"
bm25_results = bm25_search(short_query, bm25, all_chunks, n_results=5)
for chunk in bm25_results:
    print(chunk["source"], "-", chunk["text"][:100])
    print("---")

AzulSa_20170303_F-1A_EX-10.3_9943903_EX-10.3_Maintenance Agreement1 - accordance with Clause 4.5 of this Agreement. 16.2 Left intentionally blank 16.3 Suspension procedur
---
WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION AND DISTRIBUTION AGREEMENT - on business; or (b) any analogous event happens to the other party in any jurisdiction in which it i
---
THERAVANCEBIOPHARMA,INC_05_08_2020-EX-10.2-SERVICE AGREEMENT - to execute any instrument or to do anything and generally to use the Executive's name for the purpos
---
WPPPLC_04_30_2020-EX-4.28-SERVICE AGREEMENT - consent of the Company. 14.8 The Executive understands and accepts that the remuneration and benefit
---
FuseMedicalInc_20190321_10-K_EX-10.43_11575454_EX-10.43_Distributor Agreement - equivalent or similar to any of the events mentioned in clause 11.2(d) to clause 11.2(j) (inclusive)
---


In [28]:
print (bm25_results[0]["text"])

accordance with Clause 4.5 of this Agreement. 16.2 Left intentionally blank 16.3 Suspension procedure: notwithstanding the terms of Clause 16.4 below, in the event of a Company's Default as per Clause 16.1.b), the Repairer shall be entitled to suspend all or part of this Agreement by way of Notice of suspension which shall specify: (i) the Services for which such suspension shall be immediately effective until such Company's Default is corrected; and (ii) that any pending Work Order and/or placed as from the Notice of suspension will be provided upon specific commercial proposalsubject to "Payment In Advance" procedure (and/or any additional conditions to be agreed upon by the Parties, as relevant). For the sake of clarity, such Notice of suspension shall not be construed as a waiver by the Repairer of its rights regarding (i) the obligation of the Company to perform each and every of its obligations under this Agreement and/or (ii) the right of the Repairer to enforce each and every o

In [29]:
import importlib
import src.retrieval
importlib.reload(src.retrieval)
from src.retrieval import semantic_search, bm25_search, reciprocal_rank_fusion

semantic_results = semantic_search(rewritten, collection, n_results=10)
bm25_results = bm25_search(rewritten, bm25, all_chunks, n_results=10)

fused_scores = reciprocal_rank_fusion(semantic_results, bm25_results)

sorted_fused = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)

for chunk_id, score in sorted_fused[:10]:
    print(f"{chunk_id}: {score:.4f}")

GarrettMotionInc_20181001_8-K_EX-2.4_11364532_EX-2.4_Intellectual Property Agreement_18: 0.0167
REWALKROBOTICSLTD_07_10_2014-EX-10.2-STRATEGIC ALLIANCE AGREEMENT_4: 0.0167
IMMUNOMEDICSINC_08_07_2019-EX-10.1-PROMOTION AGREEMENT_37: 0.0164
InmodeLtd_20190729_F-1A_EX-10.9_11743243_EX-10.9_Manufacturing Agreement_11: 0.0164
ExactSciencesCorp_20180822_8-K_EX-10.1_11331629_EX-10.1_Promotion Agreement_71: 0.0161
SENMIAOTECHNOLOGYLTD_02_19_2019-EX-10.5-Collaboration Agreement_9: 0.0161
ReedsInc_20191113_10-Q_EX-10.4_11888303_EX-10.4_Development Agreement_7: 0.0159
LECLANCHÉ S.A. - JOINT DEVELOPMENT AND MARKETING AGREEMENT_2: 0.0159
ACCURAYINC_09_01_2010-EX-10.31-DISTRIBUTOR AGREEMENT_32: 0.0156
InmodeLtd_20190729_F-1A_EX-10.9_11743243_EX-10.9_Manufacturing Agreement_12: 0.0156


In [30]:
# Test get_chunks_by_ids: extract the top chunk_ids from the fused RRF scores,
# then retrieve their full chunk data (text, source, chunk_id) for re-ranking
import importlib
import src.retrieval
importlib.reload(src.retrieval)
from src.retrieval import get_chunks_by_ids

top_chunk_ids = [chunk_id for chunk_id, score in sorted_fused[:10]]

candidates = get_chunks_by_ids(top_chunk_ids, all_chunks)

print("Number of candidates:", len(candidates))
print("First candidate source:", candidates[0]["source"])
print("First candidate text:", candidates[0]["text"][:150])

Number of candidates: 10
First candidate source: GarrettMotionInc_20181001_8-K_EX-2.4_11364532_EX-2.4_Intellectual Property Agreement
First candidate text: an agreement in writing signed by a duly authorized officer of each of the Parties. Section 9.02. Termination prior to the Distribution. This Agreemen


In [31]:
# Test rerank: take the fused/retrieved candidates and re-score them
# against the original query using the cross-encoder for better accuracy
import importlib
import src.reranking
importlib.reload(src.reranking)
from src.reranking import rerank

reranked_results = rerank(rewritten, candidates, top_n=5)

print("Number of reranked results:", len(reranked_results))
for chunk in reranked_results:
    print(chunk["source"], "-", chunk["text"][:150])
    print("---")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 9115.60it/s]


Number of reranked results: 5
ExactSciencesCorp_20180822_8-K_EX-10.1_11331629_EX-10.1_Promotion Agreement - pursuant to Section 4.2(c) for the Calendar Year, or any portion of such Calendar Year in which such termination occurs. 8.9 Survival of Certain Oblig
---
ACCURAYINC_09_01_2010-EX-10.31-DISTRIBUTOR AGREEMENT - or termination of this Agreement for any reason will not release either party from any liabilities or obligations set forth herein which (i) the parti
---
ReedsInc_20191113_10-Q_EX-10.4_11888303_EX-10.4_Development Agreement - Termination. (a) Term. The term of this Agreement shall commence on the Effective Date and shall continue for the longer of the first anniversary of t
---
IMMUNOMEDICSINC_08_07_2019-EX-10.1-PROMOTION AGREEMENT - Party of any liability for any breach that occurred, or of any obligation to make payment that accrued, before or on the effective date of such termin
---
GarrettMotionInc_20181001_8-K_EX-2.4_11364532_EX-2.4_Intellectual Property Agreement - a

In [33]:
# Test generate_answer: use the reranked chunks as context, and ask
# Groq to answer the original (non-rewritten) user question based on them
import importlib
import src.generation
importlib.reload(src.generation)
from src.generation import generate_answer

final_answer = generate_answer(query="شو بند الإنهاء بهاد العقد؟", reranked_chunks=reranked_results, api_key=api_key)

print(final_answer)

**بند الإنهاء في العقد (المقتطفات المتوفرة)**  

1. **مدة العقد (Term)**  
   - يبدأ العقد في تاريخ النفاذ ويستمر لمدى أطول من:  
     a) أول عام من تاريخ النفاذ، أو  
     b) مدة اتفاقية التصنيع والتوزيع (Manufacturing and Distribution Agreement).  
   - يمكن تمديد المدة باتفاق خطي بين الطرفين.

2. **الإنهاء المبكر (Early Termination)**  
   يمكن لأي طرف إنهاء العقد في أي وقت إذا تحقق أحد الشروط التالية:  

   | رقم | الشرط | الإجراءات المتوقعة |  
   |-----|--------|---------------------|  
   | (i) | عدم امتثال الطرف الآخر لأي شرط أو التزام في العقد، ولا يُعالج ذلك خلال 30 يوماً من إشعار كتابي يحدد عدم الامتثال. | كتابة إشعار بالإنهاء. |  
   | (ii) | يصبح الطرف الآخر غير سائل (insolvent)، أو يعيد تنظيمه، أو يصفٍّ. | كتابة إشعار بالإنهاء. |  
   | (iii) | يقوم الطرف الآخر بعمل تحويل (assignment) لمصلحة دائنين الشركة. | كتابة إشعار بالإنهاء. |  
   | (iv) | يُعين قاضي (receiver) لملكيات الطرف الآخر. | كتابة إشعار بالإنهاء. |  
   | (v) | تنتهي اتفاقية التصنيع والتوزيع قبل أول عام من 

In [34]:
# Test the full end-to-end pipeline with a single function call
import importlib
import src.pipeline
importlib.reload(src.pipeline)
from src.pipeline import ask_question

answer = ask_question(
    query="شو بند الإنهاء بهاد العقد؟",
    collection=collection,
    bm25=bm25,
    all_chunks=all_chunks,
    api_key=api_key
    )

print(answer)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6387.64it/s]


**بند الإنهاء (Termination Clause) في العقد:**

1. **حق الإنهاء من أي طرف**  
   - وفقاً للفقرة *«2. Termination»* يحق لكل طرف إنهاء الاتفاقية في أي وقت ودون ضرورة أن يُذكر سبب.  
   - ينص ذلك على أن انتهاء أو إنهاء الاتفاق لا يقلل أي التزام كان قد نشأ قبل تاريخ الانتهاء أو الإنهاء.

2. **إنهاء قبل «التوزيع»**  
   - §9.02 يوضح أن Honeywell يملك حق إنهاء الاتفاقية في أي وقت، كما هو رابع على أنه سيختفي تلقائياً عند انتهاء اتفاقية الانفصال (Separation Agreement).  
   - لا تُفرض أية مسؤولية أو التزام بعد هذا الإنهاء، ما عدا تلك التي تُقيد بالفقرة 9.03.

3. **أثر الإنهاء – عدم التزامات لاحقة**  
   - §9.03 يُحدد أن في حال الإنهاء قبل التوزيع، لا يتعين على أي طرف (أو أفراده أو مديريهم) تحمل أي مسؤولية أو التزام إضافي.  
   - ومع ذلك، تُظل بعض المواد (مثل ARTICLE I, VI, VII, 9.03 و ARTICLE XI) سارية بعد الإنهاء.

4. **الإنهاء نتيجة تغيير السيطرة (Change of Control)**  
   - §5.4 يُمنح حق الإنهاء الفوري عند حدوث تغيير في السيطرة على الطرف الآخر، مع شروط وإشعارات محددة.  
   - يُمكّن هذا البن

In [35]:
# Test the pipeline with a few different questions to check consistency
test_questions = [
    "شو بند الإنهاء بهاد العقد؟",
    "What are the payment terms?",
    "هل يوجد بند سرية بهاد الاتفاقية؟"
]

for q in test_questions:
    print("Question:", q)
    answer = ask_question(
        query=q,
        collection=collection,
        bm25=bm25,
        all_chunks=all_chunks,
        api_key=api_key
    )
    print("Answer:", answer)
    print("=" * 50)

Question: شو بند الإنهاء بهاد العقد؟


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 16668.00it/s]


Answer: **بند الإنهاء (Clause 16 – “Termination”)**

المادة 16 من العقد تُعطي الطرفين الحق في إنهاء الاتفاقية، سواءً جزئياً أو كلاً منها، لأي سبب يختاره كل طرف.  يوضح البنود التالية كيف يتم هذا الإنهاء، وما هي الآثار القانونية والعملية المرتبطة به:

| ما يُحدده البند | شرح موجز |
|-----------------|-----------|
| **القدرة على الإنهاء** | يمكن لأي طرف إنهاء الاتفاقية في أي وقت، بغض النظر عن السبب، كما ورد في المادة 2 (المدة والتجديد) والمواد ذات الصلة (مثل المادة 5.4 “تغيير السيطرة” و5.5 “حقوق إنهاء إضافية”). |
| **الإشعار** | يجب أن يتم الإنهاء من خلال إشعار كتابي يُقدَّم للطرف الآخر.  لا يلزم اتخاذ أي إجراء إضافي أو الحصول على موافقة من المحكمة؛ يَصبح الإنهاء نافذاً فور استلام الطرف الآخر للإشعار (أو خلال الفترة التي يحدَّدها الطرف الآخر إذا وُضعت). |
| **الإجراءات المتبعة** |  
  * **المادة 16.3 – الإيقاف**: إذا حدث تأخير أو عجز من جانب الشركة (حسب المادة 16.1 b)، يُحق للـ“Repairer” إيقاف الخدمات المتأثرة، مع تحديد الخدمات التي ستتوقف والملفات المرتبطة بها.  
  * **المادة 16.4 – الإن

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 7342.60it/s]


Answer: **Payment terms found in the excerpts**

| Context (extracted section) | Payment terms |
|-----------------------------|---------------|
| **Development services (Company ↔ MBE)** | – MBE must pay the Company’s standard billing rates (plus reimbursed expenses) **within 30 days** of the Company’s invoice.  <br>– No extra fees are payable unless the development adds a revenue‑generating feature outside the original scope, in which case the parties will renegotiate a fee.  <br>– MBE receives a credit (to be used against future billing) that must be applied within **12 months** of receipt of payment. |
| **LINK PLUS CORP – Sales / Proceeds agreement** | – Payments (e.g., gross proceeds minus production costs) are due **within 45 days** of the close of each calendar quarter.  <br>– If a payment is **not received within 30 days** after its due date, it bears interest at **12 % per annum, compounded monthly** until paid.  <br>– If a payment is **not received within 60 days** after its

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6575.62it/s]


Answer: نعم، تتضمن هذه الاتفاقية بنوداً صريحة للسرية.  
من بين الفقرات التي تُذكر فيها، على سبيل المثال:

- **المادة 9.2** (معلومات سرية) التي تُحدد ما يُعتبر “معلومات سرية” وتوضّح حقوق الطرفين في عدم الكشف عنها دون موافقة خطية صريحة، بالإضافة إلى الاستثناءات المسموح بها.
- **المادة 18.1** و 18.2** التي تنصّ على شروط الاستثناءات من الالتزامات السرية وتوضح ما يُعد عامًّا أو معروفاً مسبقاً أو يتم الحصول عليه من طرف ثالث بدون التزام بالسرية.
- **المادة 18.3** التي تُذكر أن الالتزامات السرية ستستمر لمدة عشر سنوات بعد انتهاء أو انتهاء الاتفاقية.
- **المواد 10 و 11** التي تُكرر متطلبات الحفاظ على السرية واستخدام المعلومات السرية فقط لغرض الاتفاقية.

لذلك، الاتفاقية تتضمن بنوداً واضحة للسرية والالتزام بحماية المعلومات السرية بين الطرفين.


In [36]:
# Test the pipeline with a few different questions to check consistency
test_questions = [
    "what is the termination clause of this contract?",
    "What are the payment terms?",
    "are there any secret clauses in this agreement?"
]

for q in test_questions:
    print("Question:", q)
    answer = ask_question(
        query=q,
        collection=collection,
        bm25=bm25,
        all_chunks=all_chunks,
        api_key=api_key
    )
    print("Answer:", answer)
    print("=" * 50)

Question: what is the termination clause of this contract?


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 15932.92it/s]


Answer: **Termination Clause (as set out in the contract excerpts you provided)**  

The contract allows either party to terminate the Agreement (or parts of it) under a range of circumstances, with a clear notice‑and‑effect procedure. The key points are:

| Section | What it says | How it works |
|---------|--------------|--------------|
| **2. Termination** | “Either party shall have the right to terminate this Agreement at any time for any reason.” | Simple notice‑only termination. The terminating party sends written notice; the Agreement ends at the date specified in that notice (or the receipt of the notice if no date is given).  |
| **16. Termination Procedure** | • The termination becomes effective upon receipt of the written notice, or upon any period the terminating party may grant. <br>• No additional action or court consent is required. <br>• The party may terminate *all or part* of the Agreement, and the notice must specify which Services are being terminated. <br>• Any Wor

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6446.35it/s]


Answer: **Payment Terms (as stated in the contract excerpts)**  

| Item | Detail |
|------|--------|
| **When invoices are issued** | IBM will send an invoice for any charge when that charge “begins or is due” as set out in the relevant Attachment C (or Service‑Option Attachment / Order Form). |
| **Amount due** | The invoice will list the exact amount the Customer must pay. |
| **Due date** | • If the invoice is received by the 10th day of the month, the payment must be made **by the end of that month**.<br>• If the invoice is received after the 10th, the payment is due **30 days from receipt** of the invoice. |
| **Currency** | All payments must be made in **United States dollars**. |
| **Taxes** | The Customer must pay (or provide the appropriate exemption documentation for) all taxes, duties, levies, or other fees imposed on the Services, except for taxes based on IBM’s net income.  Charges specified in the Agreement are exclusive of these taxes. |
| **Late‑payment & interest** | 

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 9288.04it/s]


Answer: Based on the excerpts you provided, there is no indication that the agreement contains any **secret clauses** (i.e., clauses that are hidden, undisclosed, or marked as “secret”).  
The contract does include a confidentiality provision (see the section that refers to “maintain as secret and confidential all Confidential Information”), but that simply requires the parties to keep information confidential—it is not a “secret clause” in the sense of a hidden or undisclosed provision. 


In [37]:
# Save all_chunks and bm25 to disk (pickle) so app/main.py can load them
# quickly at startup, instead of rebuilding them from scratch every time
import pickle
import os

os.makedirs("../data", exist_ok=True)

with open("../data/all_chunks.pkl", "wb") as f:
    pickle.dump(all_chunks, f)

with open("../data/bm25.pkl", "wb") as f:
    pickle.dump(bm25, f)

print("Saved successfully!")

Saved successfully!


In [38]:
import requests

response = requests.post(
    "http://localhost:8000/ask",
    json={"question": "شو بند الإنهاء بهاد العقد؟"}
)

print(response.status_code)
print(response.json())

200
{'answer': '**بند الإنهاء في العقد (المقتبس في الملاحظات)**  \n\n| رقم المقتبس | مضمون بند الإنهاء | الملاحظات الرئيسية |\n|-------------|--------------------|---------------------|\n| **1. المدة (Term)** | يبدأ العقد بتاريخ النفاذ (“Effective Date”) ويستمر لمدة 36 شهرًا، أو حتى انتهاء اتفاقية التصنيع والتوزيع، أو حتى تمديد بموجب اتفاق كتابي. | يوضح مدة العقد الأساسية. |\n| **2. حق الإنهاء لأي سبب** | يحق لأي طرف إنهاء العقد في أي وقت لأي سبب. | لا يتطلب أي شرط إضافي. |\n| **3. الإنهاء المبكر بسبب عدم الالتزام** | • عدم امتثال الطرف الآخر لأي شرط معتمد، مع إعطاءه 30 يومًا لإصلاح الانتهاك بعد إشعار كتابي.  <br>• إفلاس أو إعادة تنظيم أو تصفية الطرف الآخر. <br>• أي تحويل (assignment) لمصلحة دائنيه. <br>• تعيين مقرض (receiver) لممتلكات الشركة. | يمنح الطرف المتضرر فترة 30 يومًا لتصحيح الانتهاك، وإذا لم يحدث ذلك يُسمح بالإلغاء. |\n| **4. إنهاء التلقائي عند انتهاء اتفاقية التوزيع** | ينتهي العقد تلقائيًا عند انتهاء “Separation Agreement” أو “Manufacturing and Distribution Agreement” أو “